# 03 - Embeat Recommendation Model Training

## Mục tiêu
Notebook này trình bày quy trình huấn luyện và kiểm tra model gợi ý âm nhạc Embeat:

1. Nạp dữ liệu đã tiền xử lý.
2. Chuẩn hóa 9 đặc trưng âm học bằng `StandardScaler`.
3. Mã hóa thể loại bằng `MultiLabelBinarizer` và Sparse Matrix.
4. Huấn luyện model và lưu artifact.
5. Tải lại artifact, kiểm tra cấu trúc model và thử recommendation.

**Phạm vi thực nghiệm:** 29.985 tracks sau làm sạch, 9 audio features và 4.211 genres. Đây không phải mô hình được huấn luyện trên toàn bộ Embeat 45M.

> Ghi chú: `fit_incremental()` hiện tích lũy các batch rồi huấn luyện tập trung bằng `finalize_incremental_training()`. Đây là Batch Accumulation, chưa phải Online Learning thực sự.

Notebook được tinh gọn để mỗi cell chỉ hiển thị kết quả cần thiết; các cell audit, debug, in source code và Git diff đã được đưa ra khỏi luồng huấn luyện chính.

## 1. Khởi tạo môi trường và đường dẫn

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

# Hỗ trợ cả Google Colab và chạy local trong thư mục project.
COLAB_ROOT = Path("/content/drive/MyDrive/music_recommender_project")
LOCAL_ROOT = Path.cwd()

if (COLAB_ROOT / "src").exists():
    PROJECT_ROOT = COLAB_ROOT
elif (LOCAL_ROOT / "src").exists():
    PROJECT_ROOT = LOCAL_ROOT
else:
    raise FileNotFoundError(
        "Không tìm thấy thư mục project. Hãy mở notebook từ thư mục music_recommender_project."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.train import train_model
from src.recommender import EmbeatRecommender

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "balanced_eda.csv"
ARTIFACT_PATH = PROJECT_ROOT / "artifacts" / "embeat_recommender.joblib"
METADATA_PATH = PROJECT_ROOT / "artifacts" / "main_model_metadata.json"

assert DATA_PATH.exists(), f"Không tìm thấy dữ liệu: {DATA_PATH}"

print("Project:", PROJECT_ROOT)
print("Dataset:", DATA_PATH.name)
print("Artifact output:", ARTIFACT_PATH)
print("ENVIRONMENT: READY")

## 2. Nạp và kiểm tra dữ liệu

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Số dòng:", f"{len(df):,}")
print("Số cột:", len(df.columns))
print("Track ID duy nhất:", f"{df['track_id'].nunique():,}")
print("Dữ liệu trùng Track ID:", int(df['track_id'].duplicated().sum()))

required_columns = [
    "track_id", "track_name", "artist_name", "artist_genres",
    "popularity", "danceability", "energy", "valence", "tempo",
    "acousticness", "instrumentalness", "speechiness", "liveness", "loudness"
]
missing = [col for col in required_columns if col not in df.columns]
assert not missing, f"Thiếu cột bắt buộc: {missing}"

display(df.head(5))
print("DATA VALIDATION: PASS")

## 3. Huấn luyện model và lưu artifact

In [ ]:
model = train_model(
    input_path=DATA_PATH,
    output_path=ARTIFACT_PATH
)

assert ARTIFACT_PATH.exists(), "Model artifact chưa được tạo."

print("TRAINING: COMPLETED")
print("Tracks sau làm sạch:", f"{len(model.df):,}")
print("Audio matrix:", model.feature_matrix.shape)
print("Genre matrix:", model.genre_matrix.shape)
print("Artifact size (MB):", round(ARTIFACT_PATH.stat().st_size / (1024 ** 2), 2))

## 4. Kiểm tra metadata huấn luyện

In [ ]:
# train_model có thể tạo training_metadata.json.
training_metadata_path = ARTIFACT_PATH.parent / "training_metadata.json"

if training_metadata_path.exists():
    with open(training_metadata_path, "r", encoding="utf-8") as file:
        metadata = json.load(file)
    print(json.dumps(metadata, indent=2, ensure_ascii=False))
else:
    print("Không tìm thấy training_metadata.json; kiểm tra lại cấu hình trong src/train.py.")

print("METADATA CHECK: COMPLETED")

## 5. Tải lại artifact và kiểm tra tính toàn vẹn

In [ ]:
loaded_model = EmbeatRecommender.load_artifact(str(ARTIFACT_PATH))

assert len(loaded_model.df) == len(model.df)
assert loaded_model.feature_matrix.shape == model.feature_matrix.shape
assert loaded_model.genre_matrix.shape == model.genre_matrix.shape
assert loaded_model.df["track_id"].tolist() == model.df["track_id"].tolist()

print("Loaded tracks:", f"{len(loaded_model.df):,}")
print("Audio matrix:", loaded_model.feature_matrix.shape)
print("Genre matrix:", loaded_model.genre_matrix.shape)
print("TRACK ORDER: PASS")
print("ARTIFACT RELOAD: PASS")

## 6. Kiểm tra thử chức năng gợi ý

In [ ]:
query_track_id = loaded_model.df.iloc[0]["track_id"]

recommendations = loaded_model.recommend_hybrid_mmr(
    track_id=query_track_id,
    top_k=5,
    as_dict=False
)

assert len(recommendations) == 5
assert query_track_id not in recommendations["track_id"].values

display(
    recommendations[
        ["track_name", "artist_name", "artist_genres",
         "audio_similarity", "genre_similarity",
         "hybrid_similarity", "mmr_score"]
    ]
)

print("RECOMMENDATION SMOKE TEST: PASS")

## 7. Kiểm tra Batch Accumulation (tùy chọn)

Cell này kiểm tra khả năng tiếp nhận nhiều batch. Nó tạo model thử nghiệm riêng, không ghi đè model artifact chính.

Lưu ý: thao tác này chỉ chứng minh cơ chế tích lũy batch và huấn luyện sau khi tổng hợp, không chứng minh Online Learning.

In [ ]:
import numpy as np

batch_test_model = EmbeatRecommender()
batches = np.array_split(df, 3)

for batch in batches:
    batch_test_model.fit_incremental(batch)

assert len(batch_test_model._pending_batches) == 3
received_rows = sum(len(batch) for batch in batch_test_model._pending_batches)
assert received_rows == len(df)

batch_test_model.finalize_incremental_training()

assert batch_test_model.feature_matrix.shape == (len(batch_test_model.df), 9)
assert len(batch_test_model._pending_batches) == 0

print("Received batches:", len(batches))
print("Received rows:", f"{received_rows:,}")
print("Cleaned rows:", f"{len(batch_test_model.df):,}")
print("BATCH ACCUMULATION TEST: PASS")

## 8. Tổng kết

Sau khi chạy thành công các cell chính, thư mục `artifacts/` cần có model artifact để Recommendation Engine và Backend sử dụng.

Các kiểm tra hoàn tất trong notebook:
- Dataset validation
- Model training
- Artifact save/load
- Track ID ordering
- Recommendation smoke test
- Batch Accumulation test (tùy chọn)

Các phép đánh giá định lượng và biểu đồ được thực hiện riêng trong `notebooks/04_evaluate_model.ipynb` và `src/evaluate.py`.